# Monthly synthetic market-data viewer

Choose one stock and one month, then run all cells. The notebook reads only that month's daily candle partitions and aggregates them to 15-minute bars for quick analysis. PostgreSQL and the full tick archive are not loaded.

## 1. Choose a stock and month

Use a symbol from the archive and either a month name, abbreviation, or number. Examples: `"AAPL"` and `"January"`, or `"MSFT"` and `3`.

In [1]:
SELECTED_SYMBOL = 'AAPL'
SELECTED_MONTH = 'January'
BAR_MINUTES = 15

## 2. Load the selected month

In [2]:
from __future__ import annotations

import calendar
import json
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

DATASET_NAME = 'synthetic-market-data-2026-v1'
candidates = [Path.cwd() / DATASET_NAME, Path.cwd() / 'apps/business-backend/db/seeds' / DATASET_NAME]
DATASET = next((path.resolve() for path in candidates if (path / 'manifest.json').is_file()), None)
if DATASET is None:
    searched = '\n'.join(f'  - {path.resolve()}' for path in candidates)
    raise FileNotFoundError(f'{DATASET_NAME}/manifest.json was not found. Generate the archive first. Searched:\n{searched}')

manifest = json.loads((DATASET / 'manifest.json').read_text(encoding='utf-8'))
symbols = [item['symbol'] if isinstance(item, dict) else item for item in manifest.get('symbols', [])]
symbol = str(SELECTED_SYMBOL).strip().upper()
if symbol not in symbols:
    raise ValueError(f'Unknown symbol {symbol!r}. Choose one of: {", ".join(symbols)}')

month_lookup = {}
for number in range(1, 13):
    month_lookup[str(number)] = number
    month_lookup[f'{number:02d}'] = number
    month_lookup[calendar.month_name[number].lower()] = number
    month_lookup[calendar.month_abbr[number].lower()] = number
month_key = str(SELECTED_MONTH).strip().lower()
if month_key not in month_lookup:
    raise ValueError('Month must be 1-12, a month name, or a three-letter abbreviation.')
month = month_lookup[month_key]
year = int(manifest.get('config', {}).get('year', 2026))
month_prefix = f'{year}-{month:02d}'
month_entries = [item for item in manifest.get('candle_files', []) if item['day'].startswith(month_prefix)]
if not month_entries:
    raise ValueError(f'No candle partitions are recorded for {calendar.month_name[month]} {year}.')
month_files = [str(DATASET / item['name']) for item in month_entries]
missing_files = [path for path in month_files if not Path(path).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing candle partition: {missing_files[0]}. Restore or regenerate the archive.')
if not isinstance(BAR_MINUTES, int) or BAR_MINUTES < 1 or BAR_MINUTES > 390:
    raise ValueError('BAR_MINUTES must be an integer from 1 through 390.')

db = duckdb.connect()
query = f'''
    SELECT time_bucket(INTERVAL '{BAR_MINUTES} minutes', to_timestamp(t)) AS timestamp,
           arg_min(open, t)::DOUBLE AS open, max(high)::DOUBLE AS high,
           min(low)::DOUBLE AS low, arg_max(close, t)::DOUBLE AS close,
           sum(volume)::BIGINT AS volume, sum(trade_count)::BIGINT AS trade_count
    FROM read_parquet(?)
    WHERE symbol = ?
    GROUP BY 1 ORDER BY 1
'''
candles = db.execute(query, [month_files, symbol]).fetch_df()
if candles.empty:
    raise ValueError(f'No candle rows found for {symbol} in {calendar.month_name[month]} {year}.')
candles['timestamp'] = pd.to_datetime(candles['timestamp'], utc=True).dt.tz_convert('America/Chicago')
candles['day'] = candles['timestamp'].dt.date
display(HTML(
    f'<b>{symbol}</b> | <b>{calendar.month_name[month]} {year}</b> | '
    f'{len(month_entries)} daily partitions | {len(candles):,} {BAR_MINUTES}-minute bars | '
    f'{candles.timestamp.min()} through {candles.timestamp.max()}'
))

## 3. Monthly price and volume chart

In [3]:
figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.72, 0.28])
figure.add_trace(go.Candlestick(x=candles['timestamp'], open=candles['open'], high=candles['high'],
                                 low=candles['low'], close=candles['close'], name=symbol), row=1, col=1)
figure.add_trace(go.Bar(x=candles['timestamp'], y=candles['volume'], name='Volume', marker_color='#64748b'), row=2, col=1)
figure.update_layout(title=f'{symbol}: {calendar.month_name[month]} {year}', height=700,
                     template='plotly_white', hovermode='x unified', xaxis_rangeslider_visible=False)
figure.update_yaxes(title_text='Price', row=1, col=1)
figure.update_yaxes(title_text='Volume', row=2, col=1)
figure.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 4. Daily summary

In [ ]:
daily = (candles.groupby('day', as_index=False)
         .agg(open=('open', 'first'), high=('high', 'max'), low=('low', 'min'),
              close=('close', 'last'), volume=('volume', 'sum'), trade_count=('trade_count', 'sum')))
daily['change_pct'] = (daily['close'] / daily['open'] - 1) * 100
daily['range_pct'] = (daily['high'] / daily['low'] - 1) * 100
display(daily.style.format({
    'open': '{:.2f}', 'high': '{:.2f}', 'low': '{:.2f}', 'close': '{:.2f}',
    'volume': '{:,.0f}', 'trade_count': '{:,.0f}', 'change_pct': '{:+.2f}%', 'range_pct': '{:.2f}%'
}))